# DS-Harness starter

**Build an AI system around a small language model.**

Each task is an objective in English, sometimes with CSV files, and its answer is **one number**:
a computation stated in the text, a statistic over messy tables, or a regression fitted on a
training file. A model of 1.5 billion parameters asked directly gets the files wrong, the
arithmetic wrong and the format wrong. Your project is the system around it:

- how the objective and its files are read;
- how the system decides what kind of task it faces;
- what reaches the model;
- which tools exist and how the model calls them;
- how the answer is parsed and checked;
- how the 40 seconds of each call are spent.

**An efficient system matters more than the model.** On `dev.json`, the same 1.5B model goes from
10.6 / 100 (asked directly, answer well parsed) to 46.1 with a system that reads the files
itself, shows the model a clean view of them and runs the code it writes.

This notebook runs on a free Colab **T4 GPU**: the data, the first solution in detail, a tool and
a loop as a skeleton, the directions, how to measure, how to submit. Challenge page:
[ml-arena.com/viewchallenge/194](https://ml-arena.com/viewchallenge/194).

---

## 0. Setup

**Runtime → Change runtime type → T4 GPU.** The ML-Arena client is the package **`mlarena-sdk`**
(imported as `mlarena`; `pip install mlarena` is an unrelated package). Colab already has torch,
transformers, pandas and numpy.

The code is the public repository [racousin/ds-harness](https://github.com/racousin/ds-harness):

| file | what it is |
|---|---|
| `scoring.py` | the scorer and the delivery loop (`run_agent`) the leaderboard runs |
| `schema.md` | formats, score, timing, the machine: read it once |
| `localtest.py` | runs an agent file on `dev.json` the way the leaderboard does |
| `dsh.py` | the kit: model loading, answer parsers, calculator, tool-call parser, code runner |
| `stage1_direct.py` | the first solution (section 2) |
| `stage2_tool_loop.py` | a tool and a loop, with the loop body to write (section 3) |

In [ ]:
!pip install -q mlarena-sdk accelerate
!git clone -q https://github.com/racousin/ds-harness
import os
os.chdir("ds-harness")
print(sorted(os.listdir(".")))

Your API key is on your ML-Arena **Profile** page (`mlk_user_...`). `download_dataset` writes
`dev.json`: 180 public tasks with their files, answers and types.

In [ ]:
from getpass import getpass
import mlarena

CHALLENGE_ID = 194
client = mlarena.connect(api_key=getpass("ML-Arena API key (mlk_user_...): "))
print(client.download_dataset(CHALLENGE_ID, "."))

In [ ]:
import collections, json, re, tempfile, time
import pandas as pd
import dsh, scoring

dev = json.load(open("dev.json"))
print(len(dev), "tasks")

---

## 1. The data

### What your agent receives

Your `Agent.solve(tasks)` is called with 8 tasks at a time. A task is **only** its `id`, its
`objective` and the **paths** of its files. The platform writes the files of a call to disk just
before the call and deletes them after. `scoring.write_files` does exactly that here:

In [ ]:
no_file = next(t for t in dev if not t["files"])
with_files = next(t for t in dev if t["type"] == "table.filter")
root = tempfile.mkdtemp()
payload = scoring.write_files([no_file, with_files], root)
for p in payload:
    print(json.dumps(p, indent=1), "\n")

In [ ]:
for path in payload[1]["files"]:
    print(f"--- {os.path.basename(path)}")
    print("".join(open(path).readlines()[:8]))

Read the objective of the second task again, then the files. The headers are short codes that
change from task to task; the objective names a column by its **description** in
`data_dictionary.csv`. The objective also states how to read the file: the separator, the decimal
mark, how missing values are written, values that carry their unit, duplicated rows, a value
that means *not recorded*. Getting all of this right is the system's job, not the model's.

### What only you see

`dev.json` also carries, for each task, its `answer`, its tolerance `tol` and its `type`. The
agent never gets them: your system must recognise the kind of task from the objective.

In [ ]:
df = pd.DataFrame([{"type": t["type"], "family": t["type"].split(".")[0], "files": len(t["files"])}
                   for t in dev])
df.groupby(["family", "type"]).size().rename("tasks").reset_index()

In [ ]:
for ty in ("calc.dates", "table.join", "fit.logistic"):
    t = next(t for t in dev if t["type"] == ty)
    print(f"[{ty}]  answer {t['answer']}  tol {t['tol']:g}  files {list(t['files'])}")
    print(t["objective"], "\n")

### Score and timing

- **One number per task.** The objective states the rounding: *n decimals* → tolerance
  `1.5 × 10⁻ⁿ`; *an integer* → `1e-6`. Inside it the task scores 1, otherwise 0. An exact,
  unrounded value is always inside.
- **Score = 100 × the mean over all tasks.** The private set has 120 tasks: 45 without files,
  50 tables, 25 fits, with other draws and a few table types that `dev.json` does not have.
- **Time.** `Agent()` has 60 s to load its model, then 15 calls of 8 tasks, **40 s per call**.
  A call that raises or takes longer ends the run and the deployment fails.

In [ ]:
for answer in (41.07, "41.07", 41.069, 41.08, "41,07", None):
    print(repr(answer), "->", scoring.score_answer({"answer": 41.07, "tol": 0.015}, answer))

---

## 2. Stage 1: the model, a structured answer, a parser

### One call

`dsh.load_llm` loads a model of the platform's list; `chat` takes a list of conversations and
generates all the replies in one batch (much cheaper than one call per task).

In [ ]:
llm = dsh.load_llm("Qwen/Qwen2.5-1.5B-Instruct")
reply = llm.chat([[{"role": "user", "content": no_file["objective"]}]], max_new_tokens=300)[0]
print(no_file["objective"], "\n\n---\n", reply, "\n\ngold:", no_file["answer"])

The reply is prose. The scorer wants one number. Stage 1 is three decisions: **what the
prompt asks for**, **how the answer is marked**, and **how your code reads it back**. We measure
them on 24 tasks without files (the cell takes a few minutes on a T4).

In [ ]:
calc = [t for t in dev if not t["files"]]
sample = calc[:24]

def measure(name, system, parse, max_new_tokens=512):
    convs = [[{"role": "system", "content": system}, {"role": "user", "content": t["objective"]}]
             for t in sample]
    t0 = time.time()
    replies = llm.chat(convs, max_new_tokens=max_new_tokens)
    secs = time.time() - t0
    answers = [parse(r) for r in replies]
    score = sum(scoring.score_answer(t, a if a is not None else 0)[0] for t, a in zip(sample, answers))
    print(f"{name:34s} {100 * score / len(sample):5.1f} / 100   {secs / len(sample):.2f} s per task")
    return replies

**An example last line.** Show the expected last line with an example value and read
the number after `####`.

In [ ]:
copy = measure("example last line '#### 42.5'",
               "Answer the question. End with a line like: #### 42.5",
               lambda r: dsh.last_number(r.split("####")[-1]))
print(copy[0][:300])

**Three designs.** Reason first, then mark the answer:

1. a last line `ANSWER: <number>`, read by a strict parser — or with a **fallback** to the last
   number written;
2. a JSON object `{"reasoning": ..., "answer": ...}`;
3. two calls: reason freely, then ask for the number only.

In [ ]:
TAG = ("Solve the problem. Reason step by step, writing each calculation. "
       "Then write a last line of the form ANSWER: <number>, with the number only.")
tag = measure("ANSWER line, strict", TAG, dsh.tagged_number)
_ = measure("ANSWER line, else last number", TAG, lambda r: dsh.tagged_number(r) or dsh.last_number(r))

In [ ]:
def from_json(r):
    try:
        return dsh.to_number(json.loads(re.sub(r"^```(json)?|```$", "", r.strip()).strip())["answer"])
    except Exception:
        return dsh.last_number(r)

_ = measure("JSON {reasoning, answer}",
            'Reply with JSON only: {"reasoning": "<short working>", "answer": <number>}', from_json)

In [ ]:
convs = [[{"role": "system", "content": "Solve the problem. Reason step by step."},
          {"role": "user", "content": t["objective"]}] for t in sample]
first = llm.chat(convs, max_new_tokens=512)
second = llm.chat([c + [{"role": "assistant", "content": r},
                        {"role": "user", "content": "Write only the final answer as a number, rounded as asked."}]
                   for c, r in zip(convs, first)], max_new_tokens=16)
score = sum(scoring.score_answer(t, dsh.last_number(r) or 0)[0] for t, r in zip(sample, second))
print(f"two calls: {100 * score / len(sample):.1f} / 100")

Measured on the 65 dev tasks without files on the platform's GPU (Qwen2.5-1.5B):


| design | score (/100) | format obeyed |
|---|---|---|
| example last line '#### 42.5' | 23.1 | 14/65 |
| ANSWER line, strict parser | 15.4 | 18/65 |
| ANSWER line, else the last number | 26.2 | 18/65 |
| JSON {reasoning, answer}, else the last number | 13.8 | 54/65 |
| two calls: reason, then the number only | 32.3 | 19/65 |

- **the parser is a design decision**: the same replies score 15.4 with a strict `ANSWER:`
  parser and 26.2 with a fallback to the last number;
- **the best-obeyed format is not the most accurate**: JSON is followed most often and scores
  least — writing JSON takes the place of the reasoning;
- **separating the reasoning from the answer pays**: two calls score best, for one more short
  generation.

`stage1_direct.py` uses the `ANSWER:` line with the fallback: simple, one call. Two calls is
your first measured improvement.

### Where it fails, by type

In [ ]:
answers = [dsh.tagged_number(r) or dsh.last_number(r) for r in tag]
by_type = collections.defaultdict(list)
for t, a in zip(sample, answers):
    by_type[t["type"]].append(scoring.score_answer(t, a or 0)[0])
for ty, v in sorted(by_type.items()):
    print(f"{ty:20s} {100 * sum(v) / len(v):5.1f}  ({len(v)} tasks)")

Products of large numbers, standard deviations, compound interest, logarithms, days of the
week: the model reasons correctly and computes wrongly. That is what a tool fixes (section 3).

### Never let a task end the run

One exception inside `solve` ends the run and fails the deployment. `stage1_direct.py` wraps the
generation in `try/except` and answers `0` for a task it cannot read: a wrong answer costs one
task, an exception costs all of them.

### With files: the wall

`stage1_direct.py` pastes the first lines of each file into the prompt. Run it on the platform's
test run (16 dev tasks, 2 calls) and on the table types. A T4 is about three times slower than the
platform's RTX 4090, so give each call 120 s here instead of 40.

In [ ]:
!python localtest.py stage1_direct.py --test-run --timeout 120

In [ ]:
!python localtest.py stage1_direct.py --type table.stat --type table.filter --limit 16 --timeout 120

The model never sees the data: a statistic over 300 rows cannot come from 6 lines. Measured on the
full `dev.json` on the platform's GPU, stage 1 scores **10.6** (no files 26.2,
tables 2.5, fits 0.0). `stage1_direct.py` is your first submission
(section 6).

---

## 3. Stage 2: a tool and a loop (a skeleton)

A tool is a function your code runs on the model's behalf. The model asks for it in a fixed
format, your code runs it and sends the result back, and the model continues. Six parts:

| part | in `stage2_tool_loop.py` |
|---|---|
| tool spec | the system prompt says what the calculator accepts and how to call it |
| call format | `<call tool="calculator">EXPRESSION</call>`, shown once in a worked exchange (`DEMO`); generation stops at `</call>` |
| call parser | `dsh.parse_tool_call(reply)` → `("calculator", "15.18 - 15.9")` or `None` |
| execution | `observe(call)`: `dsh.calculator`, an error returned as text, never raised |
| observation | a user message `Result: -0.72` (or `Error: ...`) appended to the conversation |
| stop rule | a reply without a call, `MAX_ROUNDS` rounds, or the clock |

The worked exchange matters: with the format only described in the system prompt, the 1.5B model
writes its own (`CALL: ...`) and guesses the result; shown once, it calls the tool (measured on the
65 tasks without files: 9.2 without the exchange, 41.5 with it).

One round, by hand:

In [ ]:
import importlib, stage2_tool_loop as s2
importlib.reload(s2)
t = next(t for t in dev if t["type"] == "calc.functions")
conv = [{"role": "system", "content": s2.SYSTEM}] + s2.DEMO + [{"role": "user", "content": t["objective"]}]
r = llm.chat([conv], max_new_tokens=256, stop=["</call>"])[0]
print(r)
call = dsh.parse_tool_call(r)
print("\ncall:", call, "\nobservation:", s2.observe(call) if call else None)

In [ ]:
if call:
    conv += [{"role": "assistant", "content": r}, {"role": "user", "content": s2.observe(call)}]
    print(llm.chat([conv], max_new_tokens=256, stop=["</call>"])[0])
print("\ngold:", t["answer"])

**Your turn.** Write the loop body marked `TODO` in `stage2_tool_loop.py` (about ten lines:
keep the reply, parse a call, append the reply and the observation, keep the conversation
active). Then compare it with stage 1 on the computation types:

In [ ]:
CALC = " ".join(f"--type {ty}" for ty in sorted({t["type"] for t in dev if not t["files"]}))
!python localtest.py stage1_direct.py {CALC} --timeout 120 | tail -14
!python localtest.py stage2_tool_loop.py {CALC} --timeout 120 | tail -14

On the platform's GPU, the solved loop takes the tasks without files from 26.2 to
35.4 with the same model. It does nothing for the tables and the fits: the model still
never reads the data.

---

## 4. Directions

No solution here: these are the design decisions your project is about.

- **Routing.** Recognise the kind of task from the objective (files or not, `train.csv`, words like
  *percentile*, *join*, *probability*) and send it to the right prompt and tools. The private set
  has table types `dev.json` does not: a router keyed on dev wordings fails there.
- **Read the files once, yourself.** Parse each CSV with the rules the objective states, link each
  column to its dictionary description, and give the model a compact, clean view: names,
  descriptions, units, types, a few values. The model then reasons about meaning, not about bytes.
- **Data tools or code execution.** Either a few tools with arguments (`stat(column, filter, ...)`),
  easy to call and to check, or the model writes pandas code that `dsh.run_python` runs
  (general, but a 1.5B model's code fails often). Measure both.
- **A fitting tool for `fit.*`.** scikit-learn is on the platform: `LinearRegression()`, and
  `LogisticRegression(C=np.inf)` for an unregularised fit (its default is regularised, and the
  tasks are not). A tool the model calls with the target, the features and the rows beats code it
  writes from scratch.
- **Checks, repair, pacing.** Is the number in a plausible range, rounded as asked, an integer when
  one is asked? Send failed code back once with its error. Two batched model rounds fit in 40 s;
  a third may not.
- **The model.** Five are mounted: `Qwen2.5-0.5B/1.5B-Instruct`, `Qwen2.5-Coder-1.5B-Instruct`,
  `DeepSeek-R1-Distill-Qwen-1.5B`, `Qwen3-1.7B`. Change it last, and measure.

Measured on `dev.json` (platform GPU), the ladder these directions lead to:

| system | model | score | no files | tables | fits |
|---|---|---|---|---|---|
| stage 1: answer marked `ANSWER:`, parsed with a fallback | Qwen2.5-1.5B | 10.6 | 26.2 | 2.5 | 0.0 |
| stage 1 | Qwen3-1.7B | 16.7 | 44.6 | 0.0 | 2.9 |
| stage 2: + a calculator in a loop | Qwen2.5-1.5B | 13.3 | 35.4 | 1.2 | 0.0 |
| stage 2 | Qwen3-1.7B | 22.2 | 60.0 | 1.2 | 0.0 |
| stage 2 with routing: the loop without files, stage 1 with files | Qwen2.5-1.5B | 13.3 | 35.4 | 1.2 | 0.0 |
| reference: files read by the system, data view, code, one repair | Qwen2.5-1.5B | 46.1 | 86.2 | 28.8 | 11.4 |
| reference | Qwen2.5-Coder-1.5B | 53.3 | 81.5 | 47.5 | 14.3 |
| reference | Qwen3-1.7B | 62.2 | 80.0 | 62.5 | 28.6 |

---

## 5. Measure

`localtest.py --out run.json` writes every task's answer, gold, score and error. Group the
failures before you change anything.

In [ ]:
!python localtest.py stage1_direct.py --test-run --timeout 120 --out run.json > /dev/null
run = json.load(open("run.json"))
d = pd.DataFrame(run["details"])
print(d.groupby("type")["score"].mean().mul(100).round(1))
d[d.score == 0][["id", "type", "answer", "gold", "error"]].head(10)

- **A ladder.** Keep one table: system version, score by family, seconds per call. One change
  per line.
- **By type.** `dev.json` has 7 to 18 tasks per type: a difference of one task is 6–14 points on a
  type. Look at families and at the whole score.
- **Error causes.** Wrong reading of a file? Wrong column? Right code, wrong rounding? A format
  error? Each cause has its own fix.
- **Your own validation set.** `dev.json` is small, and the private set has other draws: write a
  generator for the types you work on (a few lines of Python with its own gold) and measure on
  hundreds of tasks.
- **Time.** Measure seconds per call on the platform (the run's log) and keep a margin: the T4 is
  slower, the RTX 4090 faster, a long objective slower.

---

## 6. Submit

Upload your agent file **as `agent.py`**, with `dsh.py`, and choose the **PyTorch** runtime. A
deployment runs the test run (16 dev tasks), then the scored run (120 private tasks).

In [ ]:
import shutil
shutil.copy("stage1_direct.py", "agent.py")
result = client.submit(challenge_id=CHALLENGE_ID, files=["agent.py", "dsh.py"],
                       submission_name="stage 1",
                       runtime={"language": "python", "framework": "torch"},
                       wait=True, timeout_sec=1800)
print(result["status"]["status"], result["status"]["last_status_message"])

The leaderboard shows the score and its three family means. Deployments are limited per day:
test locally first, submit when a change is measured.